# AI_assignment — Customer Churn Prediction Using Machine Learning

**Artificial Intelligence — Assignment 1**

This notebook reconstructs the workflow documented in the supplied `AI_Assignment1.pdf`:
1. Data loading and understanding
2. Data preprocessing
3. Exploratory data analysis
4. Feature engineering and encoding
5. Training and testing
6. Model evaluation
7. Application / prediction

**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv` or `.xlsx`)

> Put the dataset in the same folder as this notebook before running **Run All**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve
)

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## Step 1 — Data Loading and Understanding


In [ ]:
# Load .xlsx if present; otherwise load the CSV version.
import os

xlsx_path = "WA_Fn-UseC_-Telco-Customer-Churn.xlsx"
csv_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if os.path.exists(xlsx_path):
    df = pd.read_excel(xlsx_path)
elif os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(
        "Dataset not found. Put WA_Fn-UseC_-Telco-Customer-Churn.csv "
        "or .xlsx in the same folder as this notebook."
    )

print("Dataset loaded successfully.")
display(df.head(10))


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nNumeric statistics:")
display(df.describe().round(2))

print("\nCategorical statistics:")
display(df.describe(include="object"))


In [ ]:
print("Target variable: Churn")
print(df["Churn"].value_counts())
print("\nPercentage:")
print((df["Churn"].value_counts(normalize=True) * 100).round(1))


### Initial understanding
The dataset should contain **7,043 customers and 21 columns**. `Churn` is the target variable.  
The report records approximately **73.5% No Churn** and **26.5% Churn**, so the target is imbalanced.


## Step 2 — Data Preprocessing


In [ ]:
print("Missing values:")
display(df.isnull().sum().to_frame("Missing"))

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

# TotalCharges is stored as text because some rows contain blank spaces.
blank_total = df["TotalCharges"].astype(str).str.strip().eq("")
print("Blank TotalCharges rows:", blank_total.sum())
display(df.loc[blank_total, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])


In [ ]:
# Convert TotalCharges to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# In this dataset, the 11 missing TotalCharges rows correspond to tenure=0:
# brand-new customers who have not yet been billed. Fill with 0 rather than delete them.
print("Missing TotalCharges before filling:", df["TotalCharges"].isna().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Missing TotalCharges after filling:", df["TotalCharges"].isna().sum())

# Harmonise labels.
service_cols = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
for col in service_cols:
    df[col] = df[col].replace({
        "No internet service": "No",
        "No phone service": "No"
    })

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# customerID is only an identifier and is not useful as a predictive feature.
df_clean = df.drop(columns=["customerID"]).copy()

print("Cleaned shape:", df_clean.shape)
display(df_clean.head())


In [ ]:
# IQR outlier check used in the report.
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    print(f"{col}: {n_outliers} IQR outliers")


## Step 3 — Exploratory Data Analysis (EDA)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df_clean["Churn"].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=axes[0])
axes[0].set_title("Churn Count")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Count")

pct = df_clean["Churn"].value_counts(normalize=True) * 100
axes[1].pie(pct.values, labels=[f"{i} ({v:.1f}%)" for i, v in pct.items()],
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Churn Percentage")

plt.tight_layout()
plt.show()


In [ ]:
categorical_features = [
    "Contract", "InternetService", "PaymentMethod",
    "TechSupport", "OnlineSecurity", "PaperlessBilling",
    "SeniorCitizen", "Partner", "Dependents"
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for ax, col in zip(axes, categorical_features):
    rates = (df_clean.groupby(col)["Churn"]
             .apply(lambda s: (s == "Yes").mean() * 100)
             .sort_values(ascending=False))
    sns.barplot(x=rates.index, y=rates.values, ax=ax)
    ax.set_title(f"Churn rate (%) by {col}")
    ax.set_ylabel("Churn %")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    summary = pd.crosstab(df_clean[col], df_clean["Churn"])
    summary["Churn_%"] = (
        summary.get("Yes", 0) / summary.sum(axis=1) * 100
    ).round(1)
    print(f"\n{col}")
    display(summary)


In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, col in enumerate(numeric_cols):
    sns.histplot(data=df_clean, x=col, hue="Churn", kde=True,
                 element="step", ax=axes[0, i])
    axes[0, i].set_title(f"{col} distribution by Churn")
    sns.boxplot(data=df_clean, x="Churn", y=col, ax=axes[1, i])
    axes[1, i].set_title(f"{col} by Churn")
plt.tight_layout()
plt.show()

display(df_clean.groupby("Churn")[numeric_cols].mean().round(2))


In [ ]:
corr_df = df_clean[["tenure", "MonthlyCharges", "TotalCharges"]].copy()
corr_df["Churn_num"] = df_clean["Churn"].map({"No": 0, "Yes": 1})

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()


## Step 4 — Feature Engineering and Encoding


In [ ]:
df_fe = df_clean.copy()

# Tenure groups documented in the report.
df_fe["tenure_group"] = pd.cut(
    df_fe["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

# Number of subscribed add-on services.
addon_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
df_fe["num_services"] = (df_fe[addon_cols] == "Yes").sum(axis=1)

# Avoid division by zero for brand-new customers.
df_fe["avg_monthly_spend"] = np.where(
    df_fe["tenure"] > 0,
    df_fe["TotalCharges"] / df_fe["tenure"],
    df_fe["MonthlyCharges"]
)
df_fe["charge_gap"] = df_fe["MonthlyCharges"] - df_fe["avg_monthly_spend"]

display(df_fe[[
    "tenure", "tenure_group", "num_services",
    "MonthlyCharges", "TotalCharges", "avg_monthly_spend", "charge_gap", "Churn"
]].head())


In [ ]:
print("Churn % by tenure group")
display((df_fe.groupby("tenure_group", observed=False)["Churn"]
         .apply(lambda s: (s == "Yes").mean() * 100)).round(1))

print("\nChurn % by number of services")
display((df_fe.groupby("num_services")["Churn"]
         .apply(lambda s: (s == "Yes").mean() * 100)).round(1))


In [ ]:
# Target
y = df_fe["Churn"].map({"No": 0, "Yes": 1})

# Inputs
X_raw = df_fe.drop(columns=["Churn"]).copy()

# Binary columns: map No/Yes to 0/1.
binary_cols = [
    "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "PaperlessBilling"
]
for col in binary_cols:
    X_raw[col] = X_raw[col].map({"No": 0, "Yes": 1})

# Gender is also binary in this dataset.
X_raw["gender"] = X_raw["gender"].map({"Female": 0, "Male": 1})

# One-hot encode nominal multi-category columns.
multi_cols = ["InternetService", "Contract", "PaymentMethod", "tenure_group"]
X = pd.get_dummies(X_raw, columns=multi_cols, drop_first=True, dtype=int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Non-numeric columns:", X.select_dtypes(exclude=np.number).columns.tolist())
display(X.head())


## Step 5 — Training and Testing


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set:", X_test.shape, y_test.shape)
print("Churn % in training:", round(y_train.mean() * 100, 2))
print("Churn % in testing:", round(y_test.mean() * 100, 2))


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=2000
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print("Trained:", name)


## Step 6 — Model Evaluation


In [ ]:
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = pred
    probabilities[name] = prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_test, pred),
        display_labels=["No Churn", "Churn"]
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

for name, pred in predictions.items():
    print("\n", "=" * 70)
    print(name)
    print(classification_report(y_test, pred, target_names=["No Churn", "Churn"]))


In [ ]:
# Model comparison
results_df.plot(kind="bar", figsize=(11, 5))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# ROC curves
plt.figure(figsize=(8, 6))
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()


In [ ]:
# 5-fold stratified cross-validation using F1 score.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
    cv_rows.append({
        "Model": name,
        "CV Mean F1": scores.mean(),
        "CV Std": scores.std()
    })

cv_df = pd.DataFrame(cv_rows).set_index("Model")
display(cv_df.round(4))


In [ ]:
# Select the final model by ROC-AUC, as documented in the report's comparison.
best_name = results_df["ROC-AUC"].idxmax()
best_model = models[best_name]

print("Selected final model:", best_name)
display(results_df.loc[[best_name]].round(4))


In [ ]:
# Feature importance / coefficients
rf = models["Random Forest"]
rf_importance = pd.Series(
    rf.feature_importances_, index=X.columns
).sort_values(ascending=False).head(12)

lr = models["Logistic Regression"]
lr_coef = pd.Series(
    lr.named_steps["model"].coef_[0], index=X.columns
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
rf_importance.sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Random Forest — Top 12 Feature Importances")

pd.concat([lr_coef.head(6), lr_coef.tail(6)]).plot(kind="barh", ax=axes[1])
axes[1].set_title("Logistic Regression — Strongest Coefficients")

plt.tight_layout()
plt.show()


## Step 7 — Application / Prediction


In [ ]:
# Store training columns so every new customer is encoded identically.
TRAINING_COLUMNS = X.columns.tolist()

def prepare_customer(customer):
    row = pd.DataFrame([customer])

    # Harmonise labels exactly as during training.
    for col in service_cols:
        if col in row.columns:
            row[col] = row[col].replace({
                "No internet service": "No",
                "No phone service": "No"
            })

    # Feature engineering.
    row["tenure_group"] = pd.cut(
        row["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=["0-12", "13-24", "25-48", "49-72"]
    )
    row["num_services"] = sum(
        (row[col] == "Yes").astype(int) for col in addon_cols
    )
    row["avg_monthly_spend"] = np.where(
        row["tenure"] > 0,
        row["TotalCharges"] / row["tenure"],
        row["MonthlyCharges"]
    )
    row["charge_gap"] = row["MonthlyCharges"] - row["avg_monthly_spend"]

    for col in binary_cols:
        row[col] = row[col].map({"No": 0, "Yes": 1})

    row["gender"] = row["gender"].map({"Female": 0, "Male": 1})
    row = pd.get_dummies(row, columns=multi_cols, drop_first=False, dtype=int)

    # Reindex to exactly the columns seen during training.
    row = row.reindex(columns=TRAINING_COLUMNS, fill_value=0)
    return row

def predict_churn(customer, model=best_model):
    row = prepare_customer(customer)
    probability = float(model.predict_proba(row)[0, 1])
    prediction = "CHURN" if probability >= 0.50 else "NO CHURN"
    return {
        "Churn Probability": f"{probability * 100:.1f}%",
        "Prediction": prediction
    }


In [ ]:
# Three example customers similar to those described in the report.
customer_A = {
    "gender": "Female", "SeniorCitizen": "No", "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month",
    "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check",
    "MonthlyCharges": 90.0, "TotalCharges": 180.0
}

customer_B = {
    "gender": "Male", "SeniorCitizen": "No", "Partner": "Yes", "Dependents": "Yes",
    "tenure": 68, "PhoneService": "Yes", "MultipleLines": "Yes",
    "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
    "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Two year",
    "PaperlessBilling": "No", "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 75.0, "TotalCharges": 5100.0
}

customer_C = {
    "gender": "Female", "SeniorCitizen": "Yes", "Partner": "No", "Dependents": "No",
    "tenure": 24, "PhoneService": "Yes", "MultipleLines": "Yes",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "One year",
    "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check",
    "MonthlyCharges": 100.0, "TotalCharges": 2400.0
}

for label, customer in {
    "Customer A": customer_A,
    "Customer B": customer_B,
    "Customer C": customer_C
}.items():
    print(label, "->", predict_churn(customer))


In [ ]:
# Sanity-check the final model on eight real test customers.
sample_idx = X_test.head(8).index
sample = df_fe.loc[sample_idx, ["tenure", "MonthlyCharges", "Churn"]].copy()
sample["Churn Probability"] = best_model.predict_proba(X.loc[sample_idx])[:, 1]
sample["Predicted"] = np.where(best_model.predict(X.loc[sample_idx]) == 1, "Churn", "No Churn")
sample["Actual"] = sample["Churn"].map({"Yes": "Churn", "No": "No Churn"})
sample["Correct?"] = sample["Predicted"] == sample["Actual"]
display(sample)


## Final Conclusion

This notebook implements the complete customer-churn machine-learning pipeline described in the assignment report.  
The exact metric values can vary if package versions or implementation details differ, but with the supplied dataset the workflow should reproduce the same overall analysis:

- imbalanced churn target;
- strong effects from contract type, tenure, internet service and payment method;
- comparison of Logistic Regression, Decision Tree and Random Forest;
- evaluation with Accuracy, Precision, Recall, F1 and ROC-AUC;
- cross-validation and feature importance;
- final prediction function for new customers.

### Limitations
The dataset is a single snapshot and does not contain usage trends, complaints, service-quality history, competitor offers or other real-world churn drivers. The models are also only lightly tuned.
